# Para la creación de la hipotesis haremos los siguientes pasos:

0. Traer librerias a usar y el df

1. Entendimiento de los datos

2. Entrada X, salida Y 

3. Selección de la hipotesis y Argumentos a favor y en contra

4. Preprocesamiento de datos sin fuga

5. Parametros e hiperparametros a estimar

6. Entrenamiento y evaluacion 

# 0. Librerias a usar y el df

In [4]:
from sklearn import set_config
set_config(display="diagram")

In [ ]:
import pandas as pd
import sklearn as sk
import numpy as np
import kagglehub
import os, glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score
)

c:\Users\workw\Documents\Met aprendizaje automatico\Hipotesis_aprendizaje_supervisado\Machine_Learning_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Path to dataset files:", path)

csv = glob.glob(os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))[0]
df = pd.read_csv(csv)
df.head()

Path to dataset files: C:\Users\workw\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


: 

# 1.Entendimiento de los datos

In [ ]:
print(df.info())
print("\nDistribución de Churn:")
print(df["Churn"].value_counts(normalize=True))

: 

: 

: 

# 2. Entrada X y salida Y

In [ ]:
df_ = df.copy()

y = df_["Churn"].map({"Yes": 1, "No": 0})
X = df_.drop(columns=["customerID", "Churn"])

: 

: 

: 

# 3. Selección de la hipotesis

Nuestras hipotesis seleccionadas son las siguientes:

H1: SVM es buena para realizar la clasificacion binaria con frontera y requiere un escalado numerico

H2 Random forest es robusto con los datos tabulares y puede capturar relaciones no lineales 

A favor SVM:
- Controla el sobreajuste con C y gamma; buen desempeño en problemas binarios.
- Puede capturar no-linealidad con kernel RBF.
- Con dataset tamaño medio (≈7k) es viable con pipeline correcto.
A favor Random Forest:
- Maneja bien no-linealidad e interacciones.
- Robusto al ruido y suele rendir bien en tabular.
- Permite ranking de variables (explicabilidad).
En contra MLP (no seleccionado):
- Requiere tuning de arquitectura y regularización; sensible a escalado y convergencia.
- Menos interpretable para negocio comparado con RF.

En contra KNN (no seleccionado):
- Basado en distancia: con muchas categóricas one-hot la distancia se degrada.
- Sensible a escalado y curse of dimensionality.
- Inferencia más lenta y sensible a ruido/desbalance.

# 4. Preprocesamiento de datos sin fuga


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Convertidor seguro de TotalCharges (dentro del pipeline)
class FixTotalCharges(BaseEstimator, TransformerMixin):
    def __init__(self, col="TotalCharges"):
        self.col = col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Convierte strings con espacios a numérico; lo no convertible -> NaN
        X[self.col] = pd.to_numeric(X[self.col].astype(str).str.strip(), errors="coerce")
        return X

# Columnas numéricas esperadas (incluye TotalCharges ya convertido por FixTotalCharges)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())     # crítico para SVM
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

: 

: 

: 

# 5. Parametros e hiperparametros a estimar



In [ ]:
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

svm_pipe = Pipeline(steps=[
    ("fix_totalcharges", FixTotalCharges("TotalCharges")),
    ("prep", preprocess),
    ("clf", svm_model)
])

rf_pipe = Pipeline(steps=[
    ("fix_totalcharges", FixTotalCharges("TotalCharges")),
    ("prep", preprocess),
    ("clf", rf_model)
])

: 

: 

: 

# 6. Entrenamiento y evaluacion 

In [ ]:
def evaluate_model(pipe, X_test, y_test, name="model"):
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None

    print(f"\n==================== {name} ====================")
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print("\nClassification report:\n", classification_report(y_test, pred))
    if proba is not None:
        print("ROC-AUC:", roc_auc_score(y_test, proba))

# Entrenar SOLO con train
svm_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)

# Evaluar en test (sin fuga)
evaluate_model(svm_pipe, X_test, y_test, name="SVM (RBF)")
evaluate_model(rf_pipe, X_test, y_test, name="Random Forest")


: 

: 

: 

# 7. Validación repetida

In [ ]:
sss = StratifiedShuffleSplit(n_splits=10, test_size=0.30, random_state=42)

def cv_auc(pipe, X, y, splitter, name="model"):
    aucs = []
    for tr, te in splitter.split(X, y):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        m = Pipeline(steps=pipe.steps)  # copia ligera del pipeline
        m.fit(X_tr, y_tr)
        proba = m.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, proba))

    print(f"\nAUC promedio ({name}): {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

: 

: 

: 